# Phase-3 final_test

Sealed adapter-only FINAL_TEST evaluation: nine adapters, 2,700 generations at S=3; unchanged baseline reused. Before launch run the sibling builder with `--finalize --train-root TRAIN_ROOT` after adapter freeze. Copy its FINAL_EVAL_PAYLOAD.zip and FINAL_EVAL_MANIFEST.md to Hinglish_Phase3 on Drive. The payload cannot be built before a matching adapter freeze; no FINAL_TEST bytes are embedded during implementation. With registered attrition S=1/2 this runs 3S adapters for descriptive reporting only.


In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive, files
import importlib.metadata
drive.mount('/content/drive')
STORE = Path('/content/drive/MyDrive/Hinglish_Phase3')
STORE.mkdir(parents=True, exist_ok=True)
PACKAGE = Path('/content/phase3_step2')
PACKAGE.mkdir(parents=True, exist_ok=True)
TRAIN_ROOT = STORE / 'train'
# Build FINAL_EVAL_PAYLOAD.zip locally AFTER phase3/ADAPTERS_FROZEN.md exists and verifies.
# Copy it and FINAL_EVAL_MANIFEST.md into STORE. This fixed launcher is frozen before training.
launch = (STORE / 'FINAL_EVAL_MANIFEST.md').read_text()
manifest = json.loads(launch.split('```json\n', 1)[1].split('\n```', 1)[0])
BLOB = (STORE / 'FINAL_EVAL_PAYLOAD.zip').read_bytes()
assert hashlib.sha256(BLOB).hexdigest() == manifest['payload_sha256']
with zipfile.ZipFile(io.BytesIO(BLOB)) as z:
    assert 'phase2/final_test.v3.json' in z.namelist()
    assert 'results/phase2/final_test/Phase2_FinalTest_results.zip' in z.namelist()
    assert hashlib.sha256(z.read('phase3/ADAPTERS_FROZEN.md')).hexdigest() == manifest['adapters_frozen_sha256']
    assert hashlib.sha256(z.read('phase3/IMPLEMENTATION_MANIFEST.md')).hexdigest() == manifest['implementation_manifest_sha256']
assert importlib.metadata.version('torch') == '2.11.0+cu130', 'Wrong Colab image: stop; never install torch.'
with zipfile.ZipFile(io.BytesIO(BLOB)) as archive:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in archive.namelist())
    archive.extractall(PACKAGE)
sys.path.insert(0, str(PACKAGE))
os.chdir(PACKAGE)
contract = json.loads((PACKAGE / 'phase3/step2_runtime_contract.json').read_bytes())
for path, checksum in contract['files'].items():
    assert hashlib.sha256((PACKAGE / path).read_bytes()).hexdigest() == checksum, path
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + [f'{k}=={v}' for k,v in contract['dependencies'].items() if k != 'torch'], check=True)
from t4_smoke import train_smoke as ts, train_phase3 as train, phase3_eval as ev, phase3_contract as c
assert not ts.stack_mismatches(ts.versions()), 'Stack mismatch; restart runtime if imports are stale.'
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() == 1
assert 'T4' in torch.cuda.get_device_name(0) and torch.cuda.get_device_capability(0) == (7, 5)
# Copy the authenticated companion even when a stale local copy exists.
if (STORE / 'IMPLEMENTATION_MANIFEST.md').is_file():
    shutil.copyfile(STORE / 'IMPLEMENTATION_MANIFEST.md', PACKAGE / 'phase3/IMPLEMENTATION_MANIFEST.md')
c.verify_implementation(PACKAGE)
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception:
    pass
assert c.file_sha(PACKAGE / 'phase3/IMPLEMENTATION_MANIFEST.md') == manifest['implementation_manifest_sha256']
frozen = ev.gate(PACKAGE, TRAIN_ROOT)
ev.load_reused_baseline(PACKAGE, frozen)  # Original provenance and inference equivalence, before new GPU generation.


In [ ]:
RECOVER_STALE_LOCK = False  # Confirm no other evaluator is active before setting True.
EVALUATIONS = STORE / 'final_test'
EVALUATIONS.mkdir(exist_ok=True)
try:
    for adapter in frozen['adapters']:
        folder = train.attempt_dir(TRAIN_ROOT, adapter['arm'], adapter['seed'], adapter['attempt'])
        run_dir = EVALUATIONS / train.run_name(adapter['arm'], adapter['seed'])
        args = ['evaluate', '--root', str(PACKAGE), '--train-root', str(TRAIN_ROOT), '--split', 'final_test',
                '--run-dir', str(run_dir), '--adapter-dir', str(folder / 'final_adapter'),
                '--adapter-record', str(folder / 'final_adapter.json')]
        if RECOVER_STALE_LOCK:
            args += ['--recover-stale-lock']
        subprocess.run([sys.executable, '-m', 't4_smoke.phase3_eval'] + args, check=True)
finally:
    archive = shutil.make_archive(str(STORE / 'Phase3_FinalTest_bundle'), 'zip', STORE, 'final_test')
    # Merge this bundle into the same study root as the downloaded training tree.
    with zipfile.ZipFile(archive, 'a', zipfile.ZIP_DEFLATED) as bundle:
        bundle.write(TRAIN_ROOT / 'EVALUATION_REGISTRY.jsonl', 'train/EVALUATION_REGISTRY.jsonl')
    files.download(archive)
# No response/accuracy previews. Unseal only with the frozen analysis wrapper after all required exports validate.
